# Olist Brazilian E-Commerce: Data Exploration

Business question: which customer states carry the highest satisfaction risk from
slow delivery, and how much revenue is exposed in those states.

This notebook covers the exploration behind sql/olist_analysis.sql: checking every
raw table for nulls and duplicates, confirming how the tables join, and testing two
candidate framings (retention vs delivery/geo) before picking the final question.

In [1]:
import pandas as pd

customers = pd.read_csv('../raw_data/olist_customers_dataset.csv')
orders = pd.read_csv('../raw_data/olist_orders_dataset.csv')
items = pd.read_csv('../raw_data/olist_order_items_dataset.csv')
payments = pd.read_csv('../raw_data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../raw_data/olist_order_reviews_dataset.csv')
products = pd.read_csv('../raw_data/olist_products_dataset.csv')
sellers = pd.read_csv('../raw_data/olist_sellers_dataset.csv')
geolocation = pd.read_csv('../raw_data/olist_geolocation_dataset.csv')

tables = {'customers': customers, 'orders': orders, 'order_items': items,
          'order_payments': payments, 'order_reviews': reviews, 'products': products,
          'sellers': sellers, 'geolocation': geolocation}
for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns, {df.duplicated().sum()} full duplicate rows")

customers: 99,441 rows, 5 columns, 0 full duplicate rows
orders: 99,441 rows, 8 columns, 0 full duplicate rows
order_items: 112,650 rows, 7 columns, 0 full duplicate rows
order_payments: 103,886 rows, 5 columns, 0 full duplicate rows


order_reviews: 99,224 rows, 7 columns, 0 full duplicate rows
products: 32,951 rows, 9 columns, 0 full duplicate rows
sellers: 3,095 rows, 4 columns, 0 full duplicate rows


geolocation: 1,000,163 rows, 5 columns, 261831 full duplicate rows


## Nulls check

Every table has some nulls. The question for each is whether it is a data
quality problem or an expected pattern in the business process.

In [2]:
for name, df in tables.items():
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls):
        print(f"--- {name} ---")
        print(nulls)
        print()

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

--- order_reviews ---
review_comment_title      87656
review_comment_message    58247
dtype: int64

--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64



Findings:
- orders: missing delivery timestamps line up with canceled or unshipped orders,
  not a data error.
- order_reviews: missing comment title/message just means the customer left a
  star rating with no written comment.
- products: 610 rows missing category and description together, likely delisted
  products. 2 rows missing dimensions only.

Decision: restrict delivery-speed analysis to orders with status = 'delivered',
since the others have no delivery date to measure.

## Candidate framing 1: retention

customer_id is a one-per-order key. customer_unique_id is the real repeat
customer identifier. Checking the organic repeat purchase rate before deciding
whether retention is a strong enough signal to build the whole analysis around.

In [3]:
merged = orders.merge(customers, on='customer_id', how='left')
orders_per_customer = merged.groupby('customer_unique_id')['order_id'].nunique()
repeat_rate = (orders_per_customer > 1).mean()
print(f"customers with more than 1 order: {(orders_per_customer > 1).sum()} of {orders_per_customer.shape[0]}")
print(f"repeat purchase rate: {repeat_rate*100:.2f}%")

customers with more than 1 order: 2997 of 96096
repeat purchase rate: 3.12%


A 3.12% repeat purchase rate is real, not an error, but it means a retention
question sliced by state would have a thin sample once split further. Checking
geography and delivery instead.

## Candidate framing 2: delivery speed, satisfaction, and revenue by state

This turned out to be the stronger signal and became the final business
question. See sql/olist_analysis.sql for the full analysis, this section
just confirms the pattern exists before committing to it.

In [4]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days

merged = orders.merge(customers, on='customer_id', how='left')
rev_by_order = reviews.groupby('order_id')['review_score'].mean()
merged = merged.merge(rev_by_order.rename('review_score'), on='order_id', how='left')

state_check = merged.groupby('customer_state').agg(
    orders=('order_id', 'count'),
    avg_delivery_days=('delivery_days', 'mean'),
    avg_review=('review_score', 'mean')
)
state_check = state_check[state_check['orders'] > 100].sort_values('avg_delivery_days', ascending=False)
print(state_check.head(10).round(2))

                orders  avg_delivery_days  avg_review
customer_state                                       
AM                 148              25.99        4.21
AL                 413              24.04        3.76
PA                 975              23.32        3.85
MA                 747              21.12        3.76
SE                 350              21.03        3.81
CE                1336              20.82        3.85
PB                 536              19.95        4.02
PI                 495              18.99        3.92
RO                 253              18.91        4.05
BA                3380              18.87        3.86


The slowest-shipping states with a real sample size (MA, AL, PA, CE) are also
the lowest-reviewed. This pattern held up, so it became the final business
question. Full state-level SQL analysis, the at-risk flag, and revenue
exposure numbers are in sql/olist_analysis.sql and
cleaned_data/state_delivery_satisfaction_revenue.csv.